# 03a — RM-a: Full Fine-tuning IndoBERT

Skenario **RM-a** (baseline): seluruh parameter `indobenchmark/indobert-base-p2` di-update. Ini **training #1** dari 2 training IndoBERT pada penelitian ini.

**Dijalankan di Google Colab** (GPU). Data & output di Google Drive.

### Layout Drive yang diharapkan
Unggah folder proyek ke Drive, mis. `MyDrive/IndoBERT-with-RAC/`, berisi minimal:
```
IndoBERT-with-RAC/
├── src/                  (preprocessing.py, dataset.py, modeling.py, evaluate.py)
├── dataset/
│   ├── splits/           (train.csv, val.csv, test.csv)
│   └── processed/        (metadata.json  -> class weights)
└── results/              (dibuat otomatis: checkpoints/, metrics/, figures/)
```
Sesuaikan `PROJECT_DIR` di sel Setup bila lokasinya berbeda.

## 1. Setup Colab (mount Drive, dependensi, seed, device)

In [5]:
# --- Google Colab setup ---
import os, sys, random, json, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Bukan Colab — pakai path lokal.')

# Dependensi (Colab). transformers versi Colab biasanya cukup; -U bila perlu.
if IN_COLAB:
    os.system('pip install -q -U transformers scikit-learn')

# >>> SESUAIKAN bila perlu <<<
PROJECT_DIR = Path('/content/drive/MyDrive/IndoBERT-with-RAC') if IN_COLAB else Path('..')
SRC_DIR = PROJECT_DIR / 'src'
assert SRC_DIR.exists(), f'src tidak ditemukan di {SRC_DIR} — cek PROJECT_DIR'
sys.path.insert(0, str(SRC_DIR))

import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda | Tesla T4


## 2. Konfigurasi (hyperparameter RM-a)

In [6]:
CFG = dict(
    scenario     = 'RM-a',
    model_name   = 'indobenchmark/indobert-base-p2',
    max_length   = 128,
    epochs       = 5,
    batch_size   = 16,
    lr           = 2e-5,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    use_amp      = True,
    seed         = 42,
)
DATA_DIR  = PROJECT_DIR / 'dataset' / 'splits'
META_PATH = PROJECT_DIR / 'dataset' / 'processed' / 'metadata.json'
CKPT_DIR  = PROJECT_DIR / 'results' / 'checkpoints' / 'rma'
MET_DIR   = PROJECT_DIR / 'results' / 'metrics'
FIG_DIR   = PROJECT_DIR / 'results' / 'figures'
for d in (CKPT_DIR, MET_DIR, FIG_DIR): d.mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2))

{
  "scenario": "RM-a",
  "model_name": "indobenchmark/indobert-base-p2",
  "max_length": 128,
  "epochs": 5,
  "batch_size": 16,
  "lr": 2e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "use_amp": true,
  "seed": 42
}


## 3. Data, Tokenizer, DataLoader

Input model = kolom **`text_clean`** (hasil preprocessing). Class weights dibaca dari `metadata.json`.

In [7]:
import pandas as pd
from torch.utils.data import DataLoader
from dataset import load_tokenizer, GamblingCommentDataset

train_df = pd.read_csv(DATA_DIR / 'train.csv  ')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')
print('train/val/test:', len(train_df), len(val_df), len(test_df))

tokenizer = load_tokenizer(CFG['model_name'])

def make_loader(df, shuffle):
    ds = GamblingCommentDataset(df['text_clean'], df['label'], tokenizer=tokenizer,
                                max_length=CFG['max_length'])
    return DataLoader(ds, batch_size=CFG['batch_size'], shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(train_df, True)
val_loader   = make_loader(val_df, False)
test_loader  = make_loader(test_df, False)

class_weights = json.load(open(META_PATH, encoding='utf-8'))['class_weights']
weight = torch.tensor([class_weights['0'], class_weights['1']], dtype=torch.float, device=device)
print('class weights:', weight.tolist())

train/val/test: 6588 1402 1405


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

class weights: [0.6110183596611023, 2.7518796920776367]


## 4. Model (build + resize + smart-init special token)

In [8]:
import modeling as M
import evaluate as E

model = M.build_finetune_model(tokenizer, num_labels=2, model_name=CFG['model_name']).to(device)
print('parameter:', E.count_parameters(model))   # semua trainable (~125M)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


parameter: {'total_params': 109485314, 'trainable_params': 109485314, 'trainable_pct': 100.0}


## 5. Training Loop (AMP, class-weighted loss, best by val F1-macro)

In [9]:
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from transformers import get_linear_schedule_with_warmup

criterion = nn.CrossEntropyLoss(weight=weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
total_steps = len(train_loader) * CFG['epochs']
scheduler = get_linear_schedule_with_warmup(optimizer,
                int(total_steps * CFG['warmup_ratio']), total_steps)
scaler = GradScaler(enabled=CFG['use_amp'] and device.type == 'cuda')

@torch.no_grad()
def evaluate_split(loader):
    model.eval()
    preds, gts = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device); attn = batch['attention_mask'].to(device)
        tti = batch.get('token_type_ids')
        tti = tti.to(device) if tti is not None else None
        with autocast(enabled=scaler.is_enabled()):
            logits = model(input_ids=ids, attention_mask=attn, token_type_ids=tti).logits
        preds.append(logits.argmax(1).cpu().numpy()); gts.append(batch['labels'].numpy())
    import numpy as np
    return E.classification_metrics(np.concatenate(gts), np.concatenate(preds))

history = []
best_f1 = -1.0
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
t_start = time.perf_counter()

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    running, t_ep = 0.0, time.perf_counter()
    for batch in train_loader:
        ids = batch['input_ids'].to(device); attn = batch['attention_mask'].to(device)
        tti = batch.get('token_type_ids')
        tti = tti.to(device) if tti is not None else None
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        with autocast(enabled=scaler.is_enabled()):
            # tanpa labels= agar model tak menghitung loss internal (tak berbobot);
            # loss berbobot dihitung manual via criterion (class weights)
            logits = model(input_ids=ids, attention_mask=attn, token_type_ids=tti).logits
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item() * labels.size(0)
    val_m = evaluate_split(val_loader)
    ep_time = time.perf_counter() - t_ep
    history.append({'epoch': epoch, 'train_loss': running/len(train_df),
                    'val_f1_macro': val_m['f1_macro'], 'val_acc': val_m['accuracy'],
                    'epoch_time_s': ep_time})
    print(f"epoch {epoch}/{CFG['epochs']} | loss {running/len(train_df):.4f} | "
          f"val F1-macro {val_m['f1_macro']:.4f} | acc {val_m['accuracy']:.4f} | {ep_time:.0f}s")
    if val_m['f1_macro'] > best_f1:
        best_f1 = val_m['f1_macro']
        torch.save({'model_state': model.state_dict(), 'config': CFG,
                    'val_f1_macro': best_f1, 'epoch': epoch}, CKPT_DIR / 'best.pt')
        print(f'  -> checkpoint tersimpan (val F1-macro {best_f1:.4f})')

total_train_time = time.perf_counter() - t_start
peak_mem = E.peak_gpu_mem_mb()
print(f'\nTotal training time: {total_train_time:.0f}s | peak GPU mem: {peak_mem:.0f} MB | best val F1-macro: {best_f1:.4f}')

/tmp/ipykernel_582/3718145850.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=CFG['use_amp'] and device.type == 'cuda')
/tmp/ipykernel_582/3718145850.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
/tmp/ipykernel_582/3718145850.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):


epoch 1/5 | loss 0.2207 | val F1-macro 0.9532 | acc 0.9708 | 59s
  -> checkpoint tersimpan (val F1-macro 0.9532)


/tmp/ipykernel_582/3718145850.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
/tmp/ipykernel_582/3718145850.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):


epoch 2/5 | loss 0.0780 | val F1-macro 0.9468 | acc 0.9665 | 53s


/tmp/ipykernel_582/3718145850.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
/tmp/ipykernel_582/3718145850.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):


epoch 3/5 | loss 0.0340 | val F1-macro 0.9773 | acc 0.9864 | 60s
  -> checkpoint tersimpan (val F1-macro 0.9773)


/tmp/ipykernel_582/3718145850.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
/tmp/ipykernel_582/3718145850.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):


epoch 4/5 | loss 0.0189 | val F1-macro 0.9763 | acc 0.9857 | 56s


/tmp/ipykernel_582/3718145850.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
/tmp/ipykernel_582/3718145850.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):


epoch 5/5 | loss 0.0112 | val F1-macro 0.9738 | acc 0.9843 | 55s

Total training time: 286s | peak GPU mem: 2299 MB | best val F1-macro: 0.9773


## 6. Evaluasi Test + Efisiensi + Simpan

In [10]:
# muat checkpoint terbaik
ckpt = torch.load(CKPT_DIR / 'best.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# prediksi test
import numpy as np
preds, gts = [], []
for batch in test_loader:
    ids = batch['input_ids'].to(device); attn = batch['attention_mask'].to(device)
    tti = batch.get('token_type_ids'); tti = tti.to(device) if tti is not None else None
    with torch.no_grad(), autocast(enabled=scaler.is_enabled()):
        logits = model(input_ids=ids, attention_mask=attn, token_type_ids=tti).logits
    preds.append(logits.argmax(1).cpu().numpy()); gts.append(batch['labels'].numpy())
y_pred = np.concatenate(preds); y_true = np.concatenate(gts)

metrics = E.classification_metrics(y_true, y_pred)
print('=== Test metrics (RM-a) ===')
for k in ['accuracy','f1_macro','precision_macro','recall_macro','f1_weighted','f1_class1']:
    print(f'  {k:18s}: {metrics[k]:.4f}')

E.plot_confusion_matrix(y_true, y_pred, FIG_DIR / 'rma_confusion.png', title='RM-a — Confusion Matrix')

# efisiensi
params = E.count_parameters(model)
one = next(iter(test_loader))
ids1 = one['input_ids'][:1].to(device); attn1 = one['attention_mask'][:1].to(device)
tti1 = one.get('token_type_ids'); tti1 = tti1[:1].to(device) if tti1 is not None else None
def _predict(_):
    with torch.no_grad(), autocast(enabled=scaler.is_enabled()):
        return model(input_ids=ids1, attention_mask=attn1, token_type_ids=tti1).logits
latency = E.measure_latency(_predict, None)

row = {'scenario': 'RM-a', **{k: round(v,6) for k,v in metrics.items()},
       'total_params': params['total_params'], 'trainable_params': params['trainable_params'],
       'trainable_pct': round(params['trainable_pct'],4),
       'total_train_time_s': round(total_train_time,1),
       'sec_per_epoch': round(total_train_time/CFG['epochs'],1),
       'peak_gpu_mem_mb': round(peak_mem,1), 'latency_ms_per_sample': round(latency,4),
       'best_val_f1_macro': round(best_f1,6), **{f'hp_{k}': v for k,v in CFG.items()}}
E.save_metrics_csv(row, MET_DIR / 'rma_metrics.csv')
pd.DataFrame(history).to_csv(MET_DIR / 'rma_history.csv', index=False)
print('\nEfisiensi:', {k: row[k] for k in ['trainable_params','total_train_time_s','peak_gpu_mem_mb','latency_ms_per_sample']})
print('Tersimpan: rma_metrics.csv, rma_history.csv, rma_confusion.png, checkpoints/rma/best.pt')

/tmp/ipykernel_582/923843131.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=scaler.is_enabled()):


=== Test metrics (RM-a) ===
  accuracy          : 0.9794
  f1_macro          : 0.9651
  precision_macro   : 0.9687
  recall_macro      : 0.9616
  f1_weighted       : 0.9793
  f1_class1         : 0.9428


/tmp/ipykernel_582/923843131.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=scaler.is_enabled()):



Efisiensi: {'trainable_params': 109485314, 'total_train_time_s': 285.8, 'peak_gpu_mem_mb': 2298.5, 'latency_ms_per_sample': 11.9707}
Tersimpan: rma_metrics.csv, rma_history.csv, rma_confusion.png, checkpoints/rma/best.pt


## Ringkasan

RM-a (full fine-tuning) selesai — metrik & efisiensi tersimpan di `results/`.

- **Baseline** untuk perbandingan trade-off vs RM-b (frozen) & RM-c (RAC).
- Checkpoint: `results/checkpoints/rma/best.pt` (dipilih by val F1-macro).
- **Langkah berikut**: `03b_rmb_frozen.ipynb` (training #2 — frozen encoder + head; menyimpan embedding untuk RM-c).